In [0]:
import sys, importlib
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import src.oil_http as oh
importlib.reload(oh)

EIA_KEY = dbutils.secrets.get("oil-shock", "eia-api-key")
oh.register_secret(EIA_KEY)
BASE = "https://api.eia.gov/v2"

# (API section, frequency, series code, what it is)
SERIES = [
    # Daily spot prices
    ("petroleum/pri/spt", "daily", "RBRTE", "Brent physical spot"),
    ("petroleum/pri/spt", "daily", "RWTC", "WTI spot (Cushing)"),
    ("petroleum/pri/spt", "daily", "EER_EPMRU_PF4_Y35NY_DPG", "Gasoline spot, New York Harbor"),
    ("petroleum/pri/spt", "daily", "EER_EPD2DXL0_PF4_Y35NY_DPG", "Ultra-low sulfur diesel spot, New York Harbor"),
    ("petroleum/pri/spt", "daily", "EER_EPJK_PF4_RGC_DPG", "Jet fuel spot, US Gulf Coast"),
    ("petroleum/pri/spt", "daily", "EER_EPD2F_PF4_Y35NY_DPG", "Heating oil spot, New York Harbor"),
    # Daily futures (NYMEX WTI curve)
    ("petroleum/pri/fut", "daily", "RCLC1", "WTI futures, contract 1"),
    ("petroleum/pri/fut", "daily", "RCLC2", "WTI futures, contract 2"),
    ("petroleum/pri/fut", "daily", "RCLC3", "WTI futures, contract 3"),
    ("petroleum/pri/fut", "daily", "RCLC4", "WTI futures, contract 4"),
    # Natural gas
    ("natural-gas/pri/fut", "daily", "RNGWHHD", "Henry Hub natural gas spot"),
    # Weekly US supply
    ("petroleum/sum/sndw", "weekly", "WCESTUS1", "US commercial crude stocks"),
    ("petroleum/sum/sndw", "weekly", "WCSSTUS1", "US strategic petroleum reserve"),
    ("petroleum/sum/sndw", "weekly", "WGTSTUS1", "US gasoline stocks"),
    ("petroleum/sum/sndw", "weekly", "WDISTUS1", "US distillate stocks"),
    ("petroleum/sum/sndw", "weekly", "WCRFPUS2", "US crude production"),
    ("petroleum/sum/sndw", "weekly", "WPULEUS3", "US refinery utilization"),
    ("petroleum/sum/sndw", "weekly", "WRPUPUS2", "US total product supplied (demand)"),
    ("petroleum/sum/sndw", "weekly", "WCRIMUS2", "US crude imports"),
    ("petroleum/sum/sndw", "weekly", "WCREXUS2", "US crude exports"),
    # Monthly drilling
    ("petroleum/crd/drill", "monthly", "E_ERTRRO_XR0_NUS_C", "US rotary rigs in operation"),
    # EIA forecasts (STEO)
    ("steo", "monthly", "BREPUUS", "STEO: Brent price"),
    ("steo", "monthly", "WTIPUUS", "STEO: WTI price"),
    ("steo", "monthly", "PAPR_WORLD", "STEO: world liquids production"),
    ("steo", "monthly", "PATC_WORLD", "STEO: world liquids consumption"),
    ("steo", "monthly", "PAPR_OPEC", "STEO: OPEC liquids production"),
    ("steo", "monthly", "T3_STCHANGE_WORLD", "STEO: world inventory change"),
]

dl = oh.Downloader("eia", backoff_seconds=3)
results = []
for route, freq, code, label in SERIES:
    params = {
        "api_key": EIA_KEY, "frequency": freq, "data[0]": "value",
        "facets[series][]": code,
        "sort[0][column]": "period", "sort[0][direction]": "desc", "length": 1,
    }
    r = dl.get(f"check/{route}", f"{BASE}/{route}/data/", params=params, raise_on_error=False)
    if r is None:
        results.append((code, label, "ERROR", "", "", ""))
        continue
    body = r.json().get("response", {})
    rows = body.get("data", [])
    total = body.get("total", 0)
    dl.note_rows(len(rows), outcome="ok" if rows else "no_data")
    if rows:
        x = rows[0]
        results.append((code, label, "OK", x.get("period"), x.get("value"), f"{int(total):,}"))
    else:
        results.append((code, label, "NOT FOUND", "", "", ""))

dl.flush()

print(f"{'Code':<28} {'Result':<10} {'Latest':<12} {'Value':>10} {'History':>9}  What")
print("-" * 120)
for code, label, status, period, value, total in results:
    print(f"{code:<28} {status:<10} {str(period):<12} {str(value):>10} {total:>9}  {label}")
print("-" * 120)
print(f"{sum(r[2] == 'OK' for r in results)} of {len(results)} series found")

In [0]:
STEO = [
    ("BREPUUS", "STEO: Brent price"),
    ("WTIPUUS", "STEO: WTI price"),
    ("PAPR_WORLD", "STEO: world liquids production"),
    ("PATC_WORLD", "STEO: world liquids consumption"),
    ("PAPR_OPEC", "STEO: OPEC liquids production"),
    ("T3_STCHANGE_WORLD", "STEO: world inventory change"),
]

dl = oh.Downloader("eia", backoff_seconds=3)
print(f"{'Code':<20} {'Result':<10} {'Latest':<10} {'Value':>10}  What / error")
print("-" * 110)
for code, label in STEO:
    params = {
        "api_key": EIA_KEY, "frequency": "monthly", "data[0]": "value",
        "facets[seriesId][]": code,
        "sort[0][column]": "period", "sort[0][direction]": "desc", "length": 1,
    }
    r = dl.get("check/steo", f"{BASE}/steo/data/", params=params, raise_on_error=False)
    if r is None:
        print(f"{code:<20} {'ERROR':<10} {'':<10} {'':>10}  see error below")
        continue
    rows = r.json().get("response", {}).get("data", [])
    dl.note_rows(len(rows), outcome="ok" if rows else "no_data")
    if rows:
        x = rows[0]
        print(f"{code:<20} {'OK':<10} {x.get('period'):<10} {str(x.get('value')):>10}  {label}")
    else:
        print(f"{code:<20} {'NOT FOUND':<10} {'':<10} {'':>10}  {label}")
dl.flush()

# Show the last error message, if any
display(spark.sql("""
    SELECT dataset, status_code, error_message, called_at
    FROM oil_shock.ops.api_call_log
    WHERE source = 'eia' AND outcome = 'error'
    ORDER BY called_at DESC LIMIT 3
"""))